In [0]:
%sql
-- This creates the "Gold" folder in your data catalog
CREATE SCHEMA IF NOT EXISTS workspace.movielens_gold;

 Step 1: **Power User Identification: Identifying Power Users (Aggregation)**
**Purpose:** Summarize user activity by calculating the total number of ratings and the average score given by each user.
**Why we do this:** * **User Analysis:** Not all users are equal; some provide significantly more data than others.
* **Feature Engineering:** These counts help us identify our "critics" (Power Users). 
* **Data Reduction:** We transform 100,000+ individual rows into a summarized table of unique users, making it easier for analysts to query.
**"Binning" or "Bucketizing."** It helps us group people into levels (like "Bronze," "Silver," and "Gold" users) so the data isn't just a list of random numbers.

sum of what we finally achieved:
What we just accomplished:

    Summarization: You turned thousands of rows into a clean list of unique users.

    Feature Engineering: You created a new piece of information (user_level) that didn't exist in the raw data.

    Schema Evolution: You told Databricks how to handle a change in your table structure using overwriteSchema.
I might consider later sth like this. 
    Total Ratings Given,Math Calculation,Resulting user_level,Activity Tier
0 - 99,floor(0.xx),0,Casual Viewer
100 - 199,floor(1.xx),1,Enthusiast
200 - 299,floor(2.xx),2,Movie Buff
300 - 399,floor(3.xx),3,Power User
1000+,floor(10.xx),10,Critic / Expert


In [0]:
from pyspark.sql.functions import count, col, avg, floor

# 1. Load the clean data from the Silver layer
df_silver = spark.table("workspace.movielens_silver.fact_ratings")

# 2. Aggregate user activity
# We count their reviews and find their average score
df_power_users = (df_silver
    .groupBy("userId")
    .agg(
        count("rating").alias("total_ratings"),
        avg("rating").alias("avg_rating_given")
    )
)

# 3. Add the "User Level" logic (Binning/Grouping)
# Dividing by 100 turns 150 ratings into Level 1, 250 into Level 2, etc.
df_power_users_with_levels = df_power_users.withColumn(
    "user_level", 
    floor(col("total_ratings") / 100)
).orderBy(col("total_ratings").desc())

# 4. Save to the Gold Schema
# We use .option("overwriteSchema", "true") to allow the new column to be added
(df_power_users_with_levels.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.movielens_gold.power_users")
)

# 5. Show the final result
print("🏆 Power Users table with Levels is complete!")
display(df_power_users_with_levels.limit(10))

🏆 Power Users table with Levels is complete!


userId,total_ratings,avg_rating_given,user_level
414,2698,3.391957005189029,26
599,2478,2.6420500403551253,24
474,2108,3.398956356736243,21
448,1864,2.8473712446351933,18
274,1346,3.235884101040119,13
610,1302,3.6885560675883258,13
68,1260,3.233730158730159,12
380,1218,3.6732348111658455,12
606,1115,3.6573991031390136,11
288,1055,3.1459715639810426,10



 Step 2: **Machine Learning Preparation: Creating the ALS Training View**
**Purpose:** Create a "skinny" table containing only the essential columns: `userId`, `movieId`, and `rating`.
**Why we do this:** * **ML Optimization:** The Alternating Least Squares (ALS) recommendation algorithm is a mathematical model that only requires these three identifiers to learn patterns.
* **Performance:** Removing unnecessary text (like titles and genres) reduces the memory footprint and speeds up the training process in Phase 4.

In [0]:
# 1. Select only the "Big Three" columns required by the ALS algorithm
# userId: Who watched it?
# movieId: What did they watch?
# rating: Did they like it?
df_als_input = df_silver.select("userId", "movieId", "rating")

# 2. Save it to your Gold Schema
# We call it a 'view' or 'input' table because it's the starting line for Phase 4
df_als_input.write.mode("overwrite").saveAsTable("workspace.movielens_gold.als_training_view")

print("🤖 The ML Training View is ready! The model now has a clean 'Skinny Table' to study.")
display(df_als_input.limit(10))

🤖 The ML Training View is ready! The model now has a clean 'Skinny Table' to study.


userId,movieId,rating
610,1,5.0
608,2,2.0
608,3,2.0
600,4,1.5
604,5,3.0
610,6,5.0
606,7,2.5
501,8,3.0
599,9,1.5
609,10,4.0



Step 3: **Data Optimization: Optimization via Partitioning**
**Purpose:** Save the main fact table again, but physically organize the data on the disk by `rating_year`.
**Why we do this:** * **Query Speed:** If we only want to analyze data from the year 2020, Spark can skip all other "folders" (partitions) and only read the relevant data.
* **Cost Efficiency:** Scanning less data saves computing power and reduces costs in a production environment.

In [0]:
from pyspark.sql.functions import year

# 1. Extract the Year from our rating_date column
df_partitioned = df_silver.withColumn("rating_year", year(col("rating_date")))

# 2. Save the table, telling Spark to create physical folders for each year
# This is the "Optimization" step your teacher requested!
(df_partitioned.write
    .mode("overwrite")
    .partitionBy("rating_year")
    .saveAsTable("workspace.movielens_gold.fact_ratings_partitioned")
)

print("🚀 Optimization Complete! The data is now physically organized by Year.")

# 3. Verify the work: Show the first 10 rows to see the new year column
display(df_partitioned.limit(10))

🚀 Optimization Complete! The data is now physically organized by Year.


movieId,userId,rating,timestamp,rating_date,title,genres,release_year,pure_title,genres_array,imdbId,tmdbId,rating_year
1,610,5.0,1479542900,2016-11-19,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,Toy Story,"List(Adventure, Animation, Children, Comedy, Fantasy)",114709,862,2016
2,608,2.0,1117490786,2005-05-30,Jumanji (1995),Adventure|Children|Fantasy,1995,Jumanji,"List(Adventure, Children, Fantasy)",113497,8844,2005
3,608,2.0,1117504413,2005-05-31,Grumpier Old Men (1995),Comedy|Romance,1995,Grumpier Old Men,"List(Comedy, Romance)",113228,15602,2005
4,600,1.5,1237760055,2009-03-22,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,Waiting to Exhale,"List(Comedy, Drama, Romance)",114885,31357,2009
5,604,3.0,832080355,1996-05-14,Father of the Bride Part II (1995),Comedy,1995,Father of the Bride Part II,List(Comedy),113041,11862,1996
6,610,5.0,1493850345,2017-05-03,Heat (1995),Action|Crime|Thriller,1995,Heat,"List(Action, Crime, Thriller)",113277,949,2017
7,606,2.5,1171754710,2007-02-17,Sabrina (1995),Comedy|Romance,1995,Sabrina,"List(Comedy, Romance)",114319,11860,2007
8,501,3.0,844974090,1996-10-10,Tom and Huck (1995),Adventure|Children,1995,Tom and Huck,"List(Adventure, Children)",112302,45325,1996
9,599,1.5,1498504960,2017-06-26,Sudden Death (1995),Action,1995,Sudden Death,List(Action),114576,9091,2017
10,609,4.0,847220937,1996-11-05,GoldenEye (1995),Action|Adventure|Thriller,1995,GoldenEye,"List(Action, Adventure, Thriller)",113189,710,1996



**Project Summary: MovieLens Medallion Pipeline**
Phase 1: Bronze (The Raw Entry)


    Action: We ingested the raw MovieLens CSV files (Ratings, Movies, etc.) from Cloud Storage.

    Result: Data was saved as Delta Tables in the movielens_bronze schema.

    State: Raw, unformatted, and contains all original "messy" columns.

Phase 2: Silver (The Clean Room)

    Action: We performed data "janitor" work:

        Joined the ratings with movie titles.

        Converted Unix timestamps into human-readable rating_date.

        Cleaned up column names and handled missing values.

    Result: A master table called fact_ratings in the movielens_silver schema.

    State: High-quality, clean data ready for human analysis.

Phase 3: Gold (The Value Add) — Completed Today!

    Table 1: power_users

        Logic: Grouped by userId and counted ratings.

        Feature: Added a user_level column using floor(total_ratings / 100) to categorize users (Enthusiast, Buff, Critic).

    Table 2: als_training_view (The "Skinny Table")

        Logic: Selected ONLY userId, movieId, and rating.

        Purpose: Stripped away all text/dates to provide a lightweight, numeric-only table for the AI model to process.

    Table 3: fact_ratings_partitioned

        Logic: Physically organized data into folders by rating_year.

        Purpose: Optimized query performance for Phase 4 and beyond.

Next Objective: Phase 4 (Machine Learning)

We will teach the ALS (Alternating Least Squares) algorithm to look at the patterns in your als_training_view to predict what a user would rate a movie they haven't seen yet.
How to start on Tuesday:

    Open Databricks.

    Start your Cluster.

    Create notebook: 04_Machine_Learning_ALS.

    Copy-paste this summary to me so we can hit the ground running!
